# Noor-Edge 2.8B Training on Kaggle T4
Train the Noor-Edge PLE model using Burn framework (Rust, no Python/PyTorch).

**GPU**: T4 16GB | **Runtime**: ~5 hours for 20K steps | **Data**: HuggingFace

In [ ]:
# Step 1: Install Rust
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"
os.environ['PATH'] = f"/usr/local/cuda/bin:{os.environ['PATH']}"
os.environ['LD_LIBRARY_PATH'] = f"/usr/local/cuda/lib64:{os.environ.get('LD_LIBRARY_PATH', '')}"
!rustc --version

In [ ]:
# Step 2: Clone Noor repo
!cd /kaggle/working && git clone https://github.com/Adiuk24/Noor_-_Bangla_AI.git noor
!ls /kaggle/working/noor/Cargo.toml

In [ ]:
# Step 3: Download training data from HuggingFace
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
path = hf_hub_download(
    repo_id='Adiuk/noor-training-data',
    filename='noor_training_shards.tar.gz',
    repo_type='dataset',
    local_dir='/kaggle/working/hf_data',
    token='YOUR_HF_TOKEN_HERE'
)
print(f'Downloaded: {path}')

# Extract shards
!mkdir -p /kaggle/working/noor/data/noor_training/shards
!cd /kaggle/working/noor/data/noor_training/shards && tar xzf /kaggle/working/hf_data/noor_training_shards.tar.gz
!echo "Shards: $(ls /kaggle/working/noor/data/noor_training/shards/shard_*.bin | wc -l)"

In [ ]:
# Step 4: Build noor-burn with CUDA
!cd /kaggle/working/noor && cargo build --release -p noor-burn --features cuda --bin noor-train 2>&1 | tail -10
!echo 'BUILD DONE'

In [ ]:
# Step 5: Verify CUDA works
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!python3 -c "import ctypes; cuda=ctypes.CDLL('libcuda.so'); print('cuInit:', cuda.cuInit(0))"

In [ ]:
# Step 6: Download checkpoint from RunPod training (resume from step 10K)
from huggingface_hub import hf_hub_download
ckpt = hf_hub_download(
    repo_id='Adiuk/noor-edge-checkpoints',
    filename='noor_final.mpk',
    repo_type='model',
    local_dir='/kaggle/working/noor/checkpoints',
    token='YOUR_HF_TOKEN_HERE'
)
print(f'Checkpoint: {ckpt}')

In [ ]:
# Step 7: Start training (20K more steps)
!cd /kaggle/working/noor && cargo run --release -p noor-burn --features cuda --bin noor-train -- --config config/edge_kaggle.toml --data data/noor_training/shards/ --checkpoint-dir checkpoints/ 2>&1 | tail -100

In [ ]:
# Step 8: Upload checkpoints to HuggingFace when done
from huggingface_hub import HfApi
api = HfApi(token='YOUR_HF_TOKEN_HERE')
api.upload_folder(
    folder_path='/kaggle/working/noor/checkpoints',
    repo_id='Adiuk/noor-edge-checkpoints',
    repo_type='model'
)
print('Checkpoints uploaded to HuggingFace!')

In [ ]:
# Step 9: Quick smoke test
!cd /kaggle/working/noor && cargo run --release -p noor-burn --features cuda --bin noor-infer -- --config config/edge_kaggle.toml --checkpoint checkpoints/noor_final --prompt "1,50,100,200,500,1000" --max-tokens 20 2>&1